## Bayes' Theorem to find the conditional probability of a payroll anomaly given a specific symptom.

#### The Scenario & The Formula:
We want to find the probability that a payroll transaction is fraudulent/anomalous (A) given that it was submitted late at night (L). 
Bayes' Theorem allows us to calculate this conditional distribution P(A | L) using this formula: 
  **P(A|L) = P(L|A) * P(A)/ P(L)**

Where: 
P(A|L): Probability of an Anomaly given it was Late night (What we want to find).
P(L|A): Probability that known anomalies happen late at night.
P(A):   Base rate of anomalies in the company (Prior probability).
P(L):   Base rate of all transactions submitted late at night.

### Python example using Pandas and a mock dataset of 10,000 payroll records.
 
 This script simulates historical corporate data, calculates the required base and conditional probabilities from that data, and 
 uses Bayes' Theorem to determine the exact probability that a specific submission is anomalous.

In [9]:
import numpy as np
import pandas as pd

In [10]:
# Set seed for reproducible mock data
np.random.seed(42)

In [11]:
# ==========================================
# STEP 1: Generate Mock Payroll Dataset
# ==========================================
# We will create 10,000 payroll entries. 
# Total anomalies = ~100 (1%). Total late night submissions = ~1,050.
total_records = 10000

# 1. Generate normal records (99% of data)
normal_count = int(total_records * 0.99)
# In normal operations, only 10% of payroll is submitted late at night
normal_late_night = np.random.choice([0, 1], size=normal_count, p=[0.90, 0.10])

# 2. Generate anomalous records (1% of data)
anomaly_count = total_records - normal_count
# For fraudulent/erroneous entries, 60% happen late at night
anomaly_late_night = np.random.choice([0, 1], size=anomaly_count, p=[0.40, 0.60])

##-----------------------------------
print("total_records=", total_records)
print("normal_count=", normal_count)
print("anomaly_count=", anomaly_count)
print("-" * 40 + "\n")
##-----------------------------------

# Combine into a clean DataFrame
df = pd.DataFrame({
    'Is_Anomaly': [0] * normal_count + [1] * anomaly_count,
    'Is_Late_Night': np.concatenate([normal_late_night, anomaly_late_night])
})

# Shuffle the dataset to mimic real life
df = df.sample(frac=1).reset_index(drop=True)

print("--- DATASET SUMMARY ---")
print(df.groupby(['Is_Anomaly', 'Is_Late_Night']).size().unstack(fill_value=0))
print("-" * 40 + "\n")

total_records= 10000
normal_count= 9900
anomaly_count= 100
----------------------------------------

--- DATASET SUMMARY ---
Is_Late_Night     0    1
Is_Anomaly              
0              8949  951
1                44   56
----------------------------------------



In [12]:
# ==========================================
# STEP 2: Extract Probabilities from Data
# ==========================================

# 1. Prior Probability: P(Anomaly)
# What is the base rate of anomalies across the entire dataset?
p_anomaly = df['Is_Anomaly'].mean()

# 2. Evidence Probability: P(Late_Night)
# What percentage of all submissions happen late at night?
p_late_night = df['Is_Late_Night'].mean()

# 3. Conditional Probability: P(Late_Night | Anomaly)
# Out of the known anomalies, what fraction happened late at night?
anomalies_only = df[df['Is_Anomaly'] == 1]
p_late_given_anomaly = anomalies_only['Is_Late_Night'].mean()

##-----------------------------------
print("p_anomaly=", p_anomaly)
print("p_late_night=", p_late_night)
print("p_late_given_anomaly=", p_late_given_anomaly)
print("-" * 40 + "\n")
##-----------------------------------

p_anomaly= 0.01
p_late_night= 0.1007
p_late_given_anomaly= 0.56
----------------------------------------



In [13]:
# ==========================================
# STEP 3: Apply Bayes' Theorem
# ==========================================
# Target: P(Anomaly | Late_Night) = [P(Late_Night | Anomaly) * P(Anomaly)] / P(Late_Night)

numerator = p_late_given_anomaly * p_anomaly
denominator = p_late_night

p_anomaly_given_late_night = numerator / denominator

In [14]:
# ==========================================
# STEP 4: Print Step-by-Step Breakdown
# ==========================================
print("--- BAYESIAN STATISTICAL BREAKDOWN ---")
print(f"1. P(Anomaly)             [Base Risk Rate]         : {p_anomaly * 100:.2f}%")
print(f"2. P(Late_Night)          [Total Global Late Night]: {p_late_night * 100:.2f}%")
print(f"3. P(Late_Night | Anomaly)[Risk Signature]         : {p_late_given_anomaly * 100:.2f}%")
print("-" * 50)
print(f"👉 CONDITIONAL PROBABILITY P(Anomaly | Late_Night) : {p_anomaly_given_late_night * 100:.2f}%")

--- BAYESIAN STATISTICAL BREAKDOWN ---
1. P(Anomaly)             [Base Risk Rate]         : 1.00%
2. P(Late_Night)          [Total Global Late Night]: 10.07%
3. P(Late_Night | Anomaly)[Risk Signature]         : 56.00%
--------------------------------------------------
👉 CONDITIONAL PROBABILITY P(Anomaly | Late_Night) : 5.56%


### Step-by-Step Explanation

#### Step 1: Simulating the Environment:
We create a representative environment. Normal payroll submissions rarely happen at midnight (10% chance). 
However, bad actors or extreme workflow errors often happen outside business hours, 
meaning anomalies have a much higher rate of being filed late (60% chance).

#### Step 2: Counting Historical Frequencies
Instead of guessing numbers, the model scans the historical database df to extract three values:
  • p_anomaly(1.00%): 			Out of 10,000 files, exactly 100 are bad.
  • p_late_night(10.50%): 		Total midnight files across the company.
  • p_late_given_anomaly(60.00%):	The historical truth that 60 out of our 100 bad entries were filed late.

#### Step 3: Calculating the Conditional Shift
Without any context, if an auditor pulls a random payroll line, there is only a 1.00% chance it is broken.
But Bayes' Theorem updates this assumption based on evidence. We multiply the high probability that anomalies are late (60%) 
by the overall low rate of anomalies (1%), 
and divide it by the background noise of late-night activity (10.50%).

**The Final Result (~5.71%)**

The script will output that if a transaction is flagged as Is_Late_Night, its risk score instantly shifts from 1.00% to 5.71% 
(nearly a 600% increase in baseline risk). 
While not a absolute guarantee of fraud, it perfectly isolates a high-risk subset for your internal audit queue!